# OUARTI — Evaluation du pilote PPO V3, sans apprentissage

Modifier uniquement **REGLAGES**, puis **Tout executer**. Par defaut, le notebook liste et verifie les fichiers choisis sans connexion au jeu. `ACTIVER_EVALUATION=True` autorise la conduite reelle dans une session locale, seul en piste.

La sequence charge les modeles, verifie la session et le checkpoint CSP, restaure le depart avant chaque episode, puis utilise exclusivement `predict(..., deterministic=True)`. Aucun appel a `learn()` ou `save()` : les poids et fichiers de modeles restent inchanges. Il n'y a pas de coupure artificielle entre lots PPO pendant l'evaluation.

Le notebook exige les metadonnees V3 compatibles avec le meme circuit, la meme voiture, le meme sens et la meme etape. Si le jeu a redemarre, immobiliser la voiture au depart voulu et activer aussi `ACTIVER_SAUVEGARDE_CHECKPOINT`. Un checkpoint deja valide est conserve. Ne pas executer un autre notebook de telemetrie ou de conduite en parallele.

Le rapport distingue les echecs de conduite (sortie, collision detectee, immobilisation ou limite) des erreurs techniques. Une erreur de capteur/reset arrete la serie et conserve le rapport partiel. Les comparaisons utilisent un meme objectif et les memes penalites. Des repetitions deterministes au meme depart ne prouvent pas la generalisation a d'autres situations.

**Arret : Interrompre le noyau.** Modifier un interrupteur pendant l'execution n'arrete pas le vehicule. Garder le jeu visible pour la capture camera. Les resultats sont enregistres dans `Code/reports/evaluation_v3/<session>/`.

Le notebook est prepare pour le stage 2 (30 m). Pour reevaluer le stage 1, selectionner son modele et `ETAPE_APPRENTISSAGE=1`. Les archives `versions/.../agent` peuvent etre evaluees directement apres verification de leur empreinte.


In [ ]:
# REGLAGES — modifier cette cellule puis cliquer sur Tout executer.
ACTIVER_EVALUATION = False
ACTIVER_SAUVEGARDE_CHECKPOINT = False  # True : creer le depart manquant, voiture immobile

# Chemins relatifs a Code/models, avec ou sans .zip.
MODELES_A_EVALUER = [
    'ppo_vision_checkpoint_v3_stage2',
    # Pour revenir au stage 1 : choisir son nom ET ETAPE_APPRENTISSAGE=1.
    # Une ancienne version peut etre selectionnee via 'versions/.../agent'.
]
N_EPISODES = 3                   # premier essai stage 2, puis 30
TRACK_ATTENDU = 'ks_barcelona-layout_gp'
VOITURE_ATTENDUE = 'abarth500'
ETAPE_APPRENTISSAGE = 2          # doit correspondre aux metadonnees
DIRECTION_PROGRESSION = 1
DUREE_EPISODE_S = 25
INTERVALLE_ACTION_S = .12
MAX_ACTIONS_PAR_EPISODE = 500
PENALITE_SORTIE_PISTE = 100.0     # commune a tous les modeles compares
DEVICE_EVALUATION = 'cpu'


## Definitions — chargees automatiquement dans l ordre

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [ ]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [ ]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [ ]:
import ctypes, threading, time, numpy as np
from pathlib import Path

class VJoySafe:
    """Pilote local Windows deja valide dans 04_Test_Volant_vJoy/05_Test_Pedales."""
    X=0x30; Y=0x31; Z=0x32
    def __init__(self,device_id=1,watchdog_sec=.7):
        dll_path=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
        if not dll_path.exists(): raise FileNotFoundError(str(dll_path))
        self.dll=ctypes.CDLL(str(dll_path)); self.device_id=int(device_id)
        self.dll.GetVJDStatus.argtypes=[ctypes.c_uint]
        self.dll.GetVJDStatus.restype=ctypes.c_int
        self.dll.AcquireVJD.argtypes=[ctypes.c_uint]
        self.dll.AcquireVJD.restype=ctypes.c_int
        self.dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint]
        self.dll.SetAxis.restype=ctypes.c_int
        self.dll.RelinquishVJD.argtypes=[ctypes.c_uint]
        self.dll.RelinquishVJD.restype=None
        status=self.dll.GetVJDStatus(self.device_id)
        if status!=1: raise RuntimeError('vJoy peripherique %d indisponible, statut=%d'%(self.device_id,status))
        if not self.dll.AcquireVJD(self.device_id): raise RuntimeError('Echec AcquireVJD')
        self.lock=threading.Lock(); self.active=True
        self.last_action=time.monotonic(); self.watchdog_sec=float(watchdog_sec)
        self.last_command={'steer_cmd':0.0, 'throttle_cmd':0.0, 'brake_cmd':1.0}
        self._set_axes(16384,0,32767)  # frein de securite lors de la prise de controle
        self.thread=threading.Thread(target=self._watchdog,daemon=True)
        self.thread.start()
    def _set_axes(self,x,y,z):
        with self.lock:
            for axis,value in ((self.X,x),(self.Y,y),(self.Z,z)):
                if not self.dll.SetAxis(int(value),self.device_id,axis):
                    raise RuntimeError('Echec SetAxis vJoy sur '+hex(axis))
    def action(self,a):
        a=np.asarray(a,dtype=np.float32).reshape(-1)
        if a.shape!=(3,) or not np.isfinite(a).all(): raise ValueError('Action PPO invalide')
        steer=float(np.clip(a[0],-1.,1.))
        gas=float(np.clip(a[1],0.,1.))
        brake=float(np.clip(a[2],0.,1.))
        # Politique PPO au depart aleatoire : accelerateur et frein peuvent etre positifs.
        # Une faible demande de frein ne doit plus annuler un fort accelerateur.
        # Commandes longitudinales mutuellement exclusives : retenir la plus forte.
        if gas >= brake:
            brake = 0.0
        else:
            gas = 0.0
        self._set_axes(round((steer+1)*16383.5),round(gas*32767),round(brake*32767))
        # Ces valeurs sont les commandes *effectivement envoyees* a vJoy.
        self.last_command = {'steer_cmd':steer, 'throttle_cmd':gas, 'brake_cmd':brake}
        self.last_action=time.monotonic()
    def hold(self):
        if self.active:
            self._set_axes(16384,0,32767)
            self.last_action=time.monotonic()
    def _watchdog(self):
        while self.active:
            if time.monotonic()-self.last_action>self.watchdog_sec:
                try: self.hold()
                except Exception: pass
            time.sleep(.06)
    def close(self):
        if not self.active: return
        try:self.hold()
        finally:
            self.active=False
            self.thread.join(timeout=.3)
            self.dll.RelinquishVJD(self.device_id)

In [ ]:
# Cellule de pont entre Jupyter et l'application CSP Lua OUARTICheckpoint.
import json, os, time, uuid, math
from pathlib import Path

BRIDGE_DIR = Path.home() / 'Documents' / 'Assetto Corsa' / 'OUARTI_checkpoint_v3'
ANCHOR_FILE = BRIDGE_DIR / 'anchor.json'

class CheckpointBridge:
    def __init__(self, directory=BRIDGE_DIR):
        self.directory = Path(directory)
        self.command = self.directory / 'command.txt'
        self.response = self.directory / 'status.txt'

    def request(self, command, timeout=8.0):
        if command not in ('PING', 'STATUS', 'SAVE', 'LOAD', 'CAPABILITIES'):
            raise ValueError('Commande CSP inconnue: ' + command)
        if not self.directory.is_dir():
            raise RuntimeError('Application CSP OUARTICheckpoint absente : ' + str(self.directory))
        token = uuid.uuid4().hex
        temporary = self.directory / ('command_' + token + '.tmp')
        temporary.write_text(token + '|' + command, encoding='ascii')
        os.replace(str(temporary), str(self.command))
        deadline = time.monotonic() + float(timeout)
        last = ''
        while time.monotonic() < deadline:
            try:
                last = self.response.read_text(encoding='utf8').strip()
            except (FileNotFoundError, UnicodeError, PermissionError):
                time.sleep(0.05)
                continue
            parts = last.split('|', 2)
            if len(parts) == 3 and parts[0] == token:
                if parts[1] == 'ERROR':
                    raise RuntimeError('CSP checkpoint: ' + parts[2])
                if parts[1] == 'PENDING':
                    time.sleep(0.05)
                    continue
                return parts[1], parts[2]
            time.sleep(0.05)
        raise TimeoutError('Pas de reponse CSP pour ' + command + '. Dernier statut: ' + last[:140])

    def capabilities(self):
        code, detail = self.request('CAPABILITIES')
        if code != 'CAPABILITIES':
            raise RuntimeError('Reponse CAPABILITIES inattendue: ' + code + ' ' + detail)
        fields = {}
        for field in detail.split(';'):
            key, separator, value = field.strip().partition('=')
            if separator:
                fields[key.strip()] = value.strip()
        for key in ('offline', 'full_snapshot', 'position_fallback'):
            if fields.get(key) not in ('true', 'false'):
                raise RuntimeError('Capacite CSP absente ou invalide: ' + key + ': ' + detail)
        return {key: value == 'true' for key, value in fields.items()
                if key in ('offline', 'full_snapshot', 'position_fallback')}

    def assert_supported(self):
        caps = self.capabilities()
        if not caps['offline']:
            raise RuntimeError('Checkpoint reserve a une session locale hors ligne.')
        if not (caps['full_snapshot'] or caps['position_fallback']):
            raise RuntimeError(
                'Checkpoint indisponible: CSP ne fournit ni snapshot complet ni '
                'repositionnement autorise dans cette session. La connexion au pont fonctionne. '
                'Verifier la version CSP dans Content Manager > Settings > Custom Shaders Patch '
                '> About & Updates. Sur le PC examine le 16/11/2026, CSP 0.1.79 est installe '
                'et son SDK ne declare pas saveCarStateAsync/loadCarState. '
                'Apres mise a jour compatible avec sensors_par, relancer le jeu et refaire '
                'PING puis CAPABILITIES. Si les deux capacites restent false, ne pas lancer PPO. '
                'Le fallback position depend aussi de physics.allowed(); '
                'une mise a jour seule ne garantit pas son activation.')
        return caps

    def assert_saved(self):
        code, detail = self.request('STATUS')
        if code != 'SAVED':
            raise RuntimeError('Checkpoint non enregistre dans cette session CSP: ' + detail
                               + '. Executer la cellule de sauvegarde avec ACTIVER_SAUVEGARDE_CHECKPOINT=True, '
                               'voiture immobile, puis remettre False. Refaire SAVE apres chaque redemarrage du jeu.')
        return detail

    def load(self):
        code, detail = self.request('LOAD', timeout=8)
        if code != 'LOAD_REQUESTED':
            raise RuntimeError('CSP n a pas confirme la commande LOAD: ' + code + ' ' + detail)


def telemetry_xy(raw):
    return float(raw['world_position_x']), float(raw['world_position_y'])


def distance_from_anchor(raw, anchor):
    x, y = telemetry_xy(raw)
    xa, ya = float(anchor['x']), float(anchor['y'])
    distance = math.hypot(x - xa, y - ya)
    angle_error = abs((float(raw['yaw']) - float(anchor['yaw']) + math.pi) % (2 * math.pi) - math.pi)
    return distance, angle_error


def make_anchor_from_raw(raw, track, car):
    x, y = telemetry_xy(raw)
    if float(raw.get('speed', 999)) > 0.3:
        raise RuntimeError('Immobiliser la voiture avant sauvegarde (<1 km/h).')
    return {'track': track, 'car': car, 'x': x, 'y': y,
            'yaw': float(raw['yaw']), 'created_at': time.time(),
            'note': 'Checkpoint CSP en memoire, non reutilisable apres redemarrage du jeu.'}


In [ ]:
import os, json, time, numpy as np, gymnasium as gym
from gymnasium import spaces

class LiveAssettoVisionEnv(gym.Env):
    """1 instance = 1 jeu local, 1 connexion UDP, 1 vJoy. Jamais de simulation sans jeu."""
    metadata={'render_modes':[]}
    def __init__(self,expected_track,expected_car,episode_seconds=45,
                 action_interval=.12,progress_sign=1,log_file=None,curriculum_stage=1,offtrack_penalty=100.0):
        super().__init__()
        self.offtrack_penalty=float(offtrack_penalty)
        if not np.isfinite(self.offtrack_penalty) or self.offtrack_penalty <= 0:
            raise ValueError('La penalite de sortie doit etre un nombre positif fini.')
        self.expected_track=str(expected_track)
        self.expected_car=str(expected_car)
        self.curriculum_stage=int(curriculum_stage)
        self.goals={1:10.0,2:30.0,3:100.0,4:None}
        if self.curriculum_stage not in self.goals:
            raise ValueError('Etape du programme: 1, 2, 3 ou 4.')
        self.goal_m=self.goals[self.curriculum_stage]
        self.episode_progress=0.0
        self.rollout_remaining=None
        self.progress_sign=int(progress_sign)
        if self.progress_sign not in (-1,1): raise ValueError('progress_sign doit valoir -1 ou 1')
        self.episode_seconds=float(episode_seconds)
        self.action_interval=float(action_interval)
        self.action_space=spaces.Box(low=np.array([-1,0,0],dtype=np.float32),
                                     high=np.ones(3,dtype=np.float32),dtype=np.float32)
        self.observation_space=spaces.Dict({
            'image':spaces.Box(low=0,high=255,shape=(1,84,84),dtype=np.uint8),
            'state':spaces.Box(low=-5.,high=5.,shape=(9,),dtype=np.float32)})
        config=management('get_config'); sess=management('get_static_info')
        track=sess.get('TrackName','')
        if sess.get('TrackConfiguration'): track+='-'+str(sess['TrackConfiguration'])
        car=str(sess.get('CarName',''))
        if track!=self.expected_track or car!=self.expected_car:
            raise RuntimeError('Mauvaise session : '+track+'/'+car+' attendu '+self.expected_track+'/'+self.expected_car)
        if bool(config.get('vjoy_executed_by_server')):
            raise RuntimeError('Ce mode utilise vJoy local: laisser vjoy_executed_by_server=False.')
        if not config.get('screen_capture_enable'):
            raise RuntimeError('Capture camera desactivee dans le plugin')
        if sess.get('autoShifterOn') is False:
            raise RuntimeError('Activer la boite automatique dans Assetto Corsa avant PPO.')
        self.track=TrackGeometry(find_track_file(track))
        self.camera=CameraReader(config)
        self.sock=None; self.controller=None
        self.raw=None; self.arc=None;self.damage=0.;self.lap=0
        self.t0=0.;self.last_obs=None;self.idle_since=0.;self.episode_index=0;self.steps=0
        self.last_action=np.zeros(3,np.float32)
        self.log_file=Path(log_file) if log_file else None
        self.bridge=CheckpointBridge()
        self.bridge.request('PING', timeout=5)
        self.bridge.assert_supported()
        self.bridge.assert_saved()
        if not ANCHOR_FILE.is_file():
            raise RuntimeError('Point de depart inconnu : utiliser la cellule SAUVEGARDER CHECKPOINT avant PPO.')
        self.anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if self.anchor.get('track')!=track or self.anchor.get('car')!=car:
            raise RuntimeError('Checkpoint sauvegarde pour une autre voiture ou un autre circuit.')
        if self.log_file: self.log_file.parent.mkdir(parents=True,exist_ok=True)
        print('Config controle local: vJoy=local, plugin',config.get('sampling_freq'),'Hz, camera',config['color_mode'])
    def begin_rollout(self, steps):
        self.rollout_remaining = int(steps)
        if self.rollout_remaining <= 0:
            raise ValueError('Nombre de pas du lot invalide')

    def safe_stop(self):
        if self.controller is not None: self.controller.hold()
    def _connect(self):
        if self.controller is None:
            self.controller=VJoySafe(device_id=1)
        if self.sock is None:
            self.sock=open_telemetry()
    def _drain_pending_telemetry(self, max_packets=4096, max_seconds=0.12):
        """Vide les vieux datagrammes UDP sans bloquer ni perdre le timeout initial.

        A appeler apres le reset et avant la lecture de chaque observation :
        pendant time.sleep(), le buffer UDP conserve des mesures deja perimees.
        """
        if self.sock is None:
            return 0
        timeout_before = self.sock.gettimeout()
        removed = 0
        deadline = time.monotonic() + float(max_seconds)
        try:
            self.sock.setblocking(False)
            while removed < max_packets and time.monotonic() < deadline:
                try:
                    self.sock.recv(2**18)
                    removed += 1
                except (BlockingIOError, InterruptedError):
                    break
        finally:
            self.sock.settimeout(timeout_before)
        return removed

    def _observe(self, after_steps=None):
        """Obtient une image et une telemetrie recentes, avec delai controle.

        Si elles ne correspondent pas, on reprend une PAIRE fraiche plutot
        que de conserver la telemetrie d'un essai precedent.
        """
        deadline = time.monotonic() + 4.0
        last_error = 'Aucune paire image/telemetrie valide'
        while time.monotonic() < deadline:
            self._drain_pending_telemetry()
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                raw = next_packet(
                    self.sock,
                    timeout=min(0.8, remaining),
                    after_steps=after_steps,
                )
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                seq, stamp, frame = self.camera.read(
                    timeout=min(0.8, remaining),
                )
            except (TimeoutError, RuntimeError, OSError, ConnectionError) as exc:
                last_error = str(exc)
                continue

            packet_stamp = float(raw.get('timestamp_unix', raw['_received_unix']))
            gap = abs(float(stamp) - packet_stamp)
            if gap > 0.40:
                now = time.time()
                last_error = (
                    'Camera/telemetrie non synchronisees: ecart %.3f s '
                    '(age image %.3f s, age telemetrie %.3f s)'
                    % (gap, now - float(stamp), now - packet_stamp)
                )
                continue

            state, arc, lateral, width = numerical_state(raw, self.track)
            obs = {
                'image': frame_to_image84(frame),
                'state': state,
            }
            return raw, obs, arc, lateral, width, seq, gap
        raise TimeoutError('Impossible de synchroniser les capteurs: ' + last_error)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self._connect()
        self.safe_stop()
        self.bridge.assert_saved()
        # Ici on RESTAURE un snapshot CSP, pas un reset local sans retour au depart.
        self.bridge.load()
        time.sleep(0.45)
        self._drain_pending_telemetry()
        last_err='Pas de telemetrie valide apres restauration'
        deadline=time.monotonic()+10.0
        while time.monotonic()<deadline:
            try:
                raw,o,arc,lat,width,seq,gap=self._observe()
                distance,angle=distance_from_anchor(raw,self.anchor)
                speed=float(raw.get('speed',999.))
                if distance>1.5 or angle>0.30 or speed>0.75:
                    last_err=('Checkpoint non restaure: position %.2f m, orientation %.2f rad, '
                              'vitesse %.2f m/s' % (distance,angle,speed))
                    time.sleep(0.08)
                    continue
                break
            except (TimeoutError,RuntimeError,OSError,ConnectionError,ValueError) as e:
                last_err=str(e)
        else:
            self.safe_stop()
            raise RuntimeError('RESET FIXE REFUSE : '+last_err+'. Entrainer = False.')
        self.raw=raw;self.last_obs=o;self.arc=arc
        self.damage=float(raw.get('damage_total',0))
        self.lap=int(raw.get('LapCount',0))
        self.idle_since=time.monotonic()
        self.t0=time.monotonic();self.steps=0;self.episode_index+=1
        self.last_action=np.zeros(3,np.float32)
        self.episode_progress=0.0
        print('Reset fixe confirme: %.2f m / %.2f rad / %.2f m/s' % (distance,angle,speed))
        return o,{'camera_gap_s':gap,'camera_sequence':seq,'track':self.expected_track,
                  'reset_verified':True,'reset_distance_m':distance,'reset_yaw_error_rad':angle,
                  'curriculum_stage':self.curriculum_stage,'goal_distance_m':self.goal_m}
    def step(self,action):
        if self.sock is None or self.raw is None:
            raise RuntimeError('reset() doit etre appele avant step()')
        if self.rollout_remaining is not None:
            self.rollout_remaining -= 1
        rollout_cut = self.rollout_remaining == 0
        action=np.asarray(action,dtype=np.float32)
        try:
            self.controller.action(action)
            time.sleep(self.action_interval)
            raw,o,arc,lateral,width,seq,gap=self._observe(after_steps=self.raw.get('steps'))
            dt=float(raw.get('timestamp_unix',raw['_received_unix']))-float(self.raw.get('timestamp_unix',self.raw['_received_unix']))
            if not .005<dt<1.0: raise RuntimeError('Delta temps telemetrie invalide: %s'%dt)
            move=(arc-self.arc+self.track.length/2)%self.track.length-self.track.length/2
            move*=self.progress_sign
            if abs(move)>max(4.,100.*dt*1.7):
                # Collision, teleportation, mauvaise projection ou horodatage : episode invalide.
                raise RuntimeError('Saut de position non physique (%.1f m sur %.3f s)'%(move,dt))
            tyres=int(raw.get('numberOfTyresOut',0))
            offtrack=bool(tyres>2 or abs(lateral)>width*.5)
            new_damage=float(raw.get('damage_total',0))
            collision=bool(new_damage>self.damage+.02)
            new_lap=int(raw.get('LapCount',0))
            lap_finished=new_lap>self.lap
            now=time.monotonic(); speed=float(raw.get('speed',0))
            if speed>1.:self.idle_since=now
            idle=bool(now-self.idle_since>6.)
            wrong_way=bool(move<-.25)
            # Curriculum sans imitation : progres geometrie, puis distances croissantes.
            self.episode_progress+=float(move)
            milestone=bool(self.goal_m is not None and self.episode_progress>=self.goal_m
                           and not offtrack and not collision)
            if self.curriculum_stage==1:
                reward=float(2.5*np.clip(move,-3.,5.)-.02
                             -.006*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-25.*collision-5.*idle-2.*wrong_way
                             +20.*milestone)
            else:
                reward=float(np.clip(move,-5.,8.)-.02
                             -.015*abs(float(o['state'][4]))
                             -.008*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-35.*collision-8.*idle-2.*wrong_way
                             +30.*milestone+70.*(lap_finished and not offtrack and not collision))
            terminated=bool(offtrack or collision or idle or milestone or lap_finished)
            time_limit = bool(now-self.t0>=self.episode_seconds)
            truncated=bool(time_limit or (rollout_cut and not terminated))
            self.last_obs=o;self.raw=raw;self.arc=arc;self.damage=new_damage;self.lap=new_lap
            self.last_action=action.copy();self.steps+=1
            # Enregistrer les commandes reelles et les retours des pedales.
            # Utile pour distinguer exploration PPO et probleme de configuration du jeu.
            sent=dict(self.controller.last_command)
            def telemetry_number(*keys):
                for key in keys:
                    if key in raw and raw[key] is not None:
                        try: return float(raw[key])
                        except (ValueError,TypeError): pass
                return None
            info={'progress_m':float(move),'speed_ms':speed,'offtrack':offtrack,'collision':collision,
                  'offtrack_penalty':float(self.offtrack_penalty*offtrack),
                  'world_position_x':float(raw['world_position_x']),
                  'world_position_y':float(raw['world_position_y']),
                  'yaw':float(raw['yaw']),'lateral_m':float(lateral),'track_width_m':float(width),
                  'lap_finished':lap_finished,'idle':idle,'wrong_way':wrong_way,'camera_gap_s':gap,
                  'camera_sequence':seq,'episode_id':self.episode_index,
                  'episode_progress_m':float(self.episode_progress),
                  'milestone_reached':bool(milestone),'goal_distance_m':self.goal_m,
                  'curriculum_stage':self.curriculum_stage,
                  'rollout_cut':bool(rollout_cut and not terminated),
                  'time_limit':time_limit,
                  **sent,
                  'accStatus':telemetry_number('accStatus'),
                  'brakeStatus':telemetry_number('brakeStatus'),
                  'rpm':telemetry_number('rpm','RPM','engineRPM'),
                  'gear':telemetry_number('gear','Gear','currentGear','CurrentGear')}
            if terminated or truncated: self.safe_stop()
            if self.log_file:
                with self.log_file.open('a',encoding='utf8') as f:
                    f.write(json.dumps({'time':time.time(),'reward':reward,'terminated':terminated,
                             'truncated':truncated,**info},ensure_ascii=False)+'\n')
            return o,reward,terminated,truncated,info
        except (OSError,RuntimeError,TimeoutError,ConnectionError,ValueError) as exc:
            self.safe_stop()
            if self.last_obs is None:raise
            return self.last_obs,-40.,False,True,{'sensor_error':str(exc),'episode_id':self.episode_index}
    def close(self):
        # Even if the brake command fails, release UDP, vJoy and shared memory.
        errors=[]
        try:self.safe_stop()
        except Exception as exc:errors.append(str(exc))
        if self.sock is not None:
            try:close_telemetry(self.sock)
            except Exception as exc:errors.append(str(exc))
            finally:self.sock=None
        if self.controller is not None:
            try:self.controller.close()
            except Exception as exc:errors.append(str(exc))
            finally:self.controller=None
        for name in ('dual_shm','meta_shm'):
            mapping=getattr(getattr(self.camera,'cam',None),name,None)
            if mapping is not None:
                try:mapping.close()
                except Exception as exc:errors.append(str(exc))
        if errors:raise RuntimeError('Nettoyage evaluation : '+'; '.join(errors))


In [ ]:
def sauvegarder_checkpoint_si_absent(bridge, track, car):
    """Creation autorisee par le reglage; ne remplace pas un depart deja valide."""
    code, detail=bridge.request('STATUS')
    if code == 'SAVED' and ANCHOR_FILE.is_file():
        anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if (anchor.get('track'),anchor.get('car')) == (track,car):
            print('Checkpoint deja enregistre : conservation du depart existant.')
            return
        raise RuntimeError('Le fichier anchor.json correspond a une autre session. '
                           'Recreer volontairement le checkpoint dans une nouvelle session du jeu.')
    sock=open_telemetry()
    try:
        sample=next_packet(sock,timeout=3)
        make_anchor_from_raw(sample,track,car)  # exige une voiture immobile
        print('Sauvegarde du checkpoint CSP...')
        code, detail=bridge.request('SAVE',timeout=15)
        if code != 'SAVED':
            raise RuntimeError('Sauvegarde CSP non confirmee : '+code+' '+detail)
        sample=next_packet(sock,timeout=3)
        anchor=make_anchor_from_raw(sample,track,car)
        ANCHOR_FILE.write_text(json.dumps(anchor,indent=2),encoding='utf8')
        bridge.assert_saved()
        print('Checkpoint enregistre :',anchor)
    finally:
        close_telemetry(sock)


In [ ]:
from stable_baselines3 import PPO
from IPython.display import display
import hashlib, uuid, traceback


def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def verifier_configuration():
    if type(ACTIVER_EVALUATION) is not bool or type(ACTIVER_SAUVEGARDE_CHECKPOINT) is not bool:
        raise ValueError('Les interrupteurs doivent etre True ou False.')
    if not isinstance(MODELES_A_EVALUER,(list,tuple)) or not MODELES_A_EVALUER:
        raise ValueError('Choisir au moins un modele dans MODELES_A_EVALUER.')
    if type(N_EPISODES) is not int or N_EPISODES<=0:
        raise ValueError('N_EPISODES doit etre un entier positif.')
    if type(MAX_ACTIONS_PAR_EPISODE) is not int or MAX_ACTIONS_PAR_EPISODE<=0:
        raise ValueError('MAX_ACTIONS_PAR_EPISODE doit etre un entier positif.')
    if ETAPE_APPRENTISSAGE not in (1,2,3,4) or DIRECTION_PROGRESSION not in (-1,1):
        raise ValueError('Etape ou sens de progression invalide.')
    if not all(np.isfinite(v) and v>0 for v in (DUREE_EPISODE_S,INTERVALLE_ACTION_S,PENALITE_SORTIE_PISTE)):
        raise ValueError('Durees et penalite doivent etre positives et finies.')
    descriptors=[];seen=set()
    expected={'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
              'control':'vjoy_local_xyz_pedales_exclusives_v3',
              'observation':'dict_image_1x84x84_state_9',
              'reset':'csp_snapshot_verified','curriculum_stage':ETAPE_APPRENTISSAGE,
              'progress_sign':DIRECTION_PROGRESSION}
    for selected in MODELES_A_EVALUER:
        if not isinstance(selected,str) or not selected.strip():
            raise ValueError('Chaque modele doit etre un chemin non vide.')
        path=Path(selected)
        if not path.is_absolute():path=PROJECT/'models'/path
        if path.suffix!='.zip':path=path.with_suffix('.zip')
        path=path.resolve()
        if path in seen:raise ValueError('Modele selectionne deux fois : '+str(path))
        seen.add(path)
        meta_path=path.with_name(path.stem+'_meta.json')
        if not path.is_file() or not meta_path.is_file():
            raise FileNotFoundError('ZIP et metadonnees requis : '+str(path))
        versions=(PROJECT/'models'/'versions').resolve()
        if versions in path.parents:
            manifest_path=path.parent/'manifest.json'
            if not manifest_path.is_file():raise RuntimeError('Archive sans manifeste : '+str(path))
            manifest=json.loads(manifest_path.read_text(encoding='utf8'))
            if file_sha256(path)!=manifest.get('zip_sha256') or file_sha256(meta_path)!=manifest.get('meta_sha256'):
                raise RuntimeError('Archive alteree : '+str(path))
        meta=json.loads(meta_path.read_text(encoding='utf8'))
        mismatch={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}
        if mismatch:raise RuntimeError('Modele incompatible '+path.name+' : '+str(mismatch))
        descriptors.append({'label':str(selected),'path':path,'meta_path':meta_path,
                            'meta':meta,'sha256':file_sha256(path),
                            'meta_sha256':file_sha256(meta_path)})
    return descriptors


def construire_resume(rows):
    if not rows:return pd.DataFrame()
    data=pd.DataFrame(rows);summaries=[]
    for model,group in data.groupby('modele',sort=False):
        valid=group[group['valide']]
        successes=valid[valid['succes']]
        def rate(key):return float(valid[key].mean()*100) if len(valid) else None
        summaries.append({'modele':model,'episodes':len(group),'episodes_valides':len(valid),
            'erreurs_techniques':int((~group['valide']).sum()),
            'reussite_pct':rate('succes'),'sortie_pct':rate('offtrack'),
            'collision_detectee_pct':rate('collision'),'immobilisation_pct':rate('idle'),
            'limite_pct':rate('limite'),
            'temps_median_reussite_s':float(successes.duree_s.median()) if len(successes) else None,
            'progression_moyenne_m':float(valid.progression_m.mean()) if len(valid) else None,
            'reward_moyenne':float(valid.reward.mean()) if len(valid) else None,
            'volant_sature_pct':float(valid.volant_sature_pas.sum()/valid.pas.sum()*100) if valid.pas.sum() else None,
            'variation_volant_moyenne':float(valid.variation_volant_totale.sum()/valid.variations_comptees.sum()) if valid.variations_comptees.sum() else None,
            'vitesse_max_kmh':float(valid.vitesse_max_kmh.max()) if len(valid) else None})
    return pd.DataFrame(summaries)


def exporter_evaluation(directory,rows,state):
    directory=Path(directory)
    pd.DataFrame(rows).to_csv(directory/'episodes.csv',index=False)
    summary=construire_resume(rows)
    summary.to_csv(directory/'comparaison.csv',index=False)
    (directory/'etat_session.json').write_text(json.dumps(state,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf8')
    return summary


In [ ]:
def executer_evaluation():
    descriptors=verifier_configuration()
    print('Modeles verifies :')
    for desc in descriptors:
        print(' -',desc['label'],'| compteur apprentissage :',desc['meta'].get('total_timesteps'))
    if not (ACTIVER_EVALUATION or ACTIVER_SAUVEGARDE_CHECKPOINT):
        print('Evaluation desactivee : aucune connexion au jeu, aucune commande.')
        print('Regler ACTIVER_EVALUATION=True puis Tout executer pour mesurer le pilotage.')
        return None
    # Charger tous les modeles avant toute commande vehicule. Aucun learn/save.
    agents=[]
    if ACTIVER_EVALUATION:
        for desc in descriptors:
            agent=PPO.load(str(desc['path']),device=DEVICE_EVALUATION)
            agent.policy.set_training_mode(False)
            if int(agent.num_timesteps)!=int(desc['meta'].get('total_timesteps',-1)):
                raise RuntimeError('Compteur ZIP/metadonnees incoherent : '+desc['label'])
            agents.append(agent)
    bridge=CheckpointBridge()
    print('Application :',bridge.request('PING'))
    print('Capacites :',bridge.assert_supported())
    session=management('get_static_info')
    track=session.get('TrackName','')
    if session.get('TrackConfiguration'):track+='-'+str(session['TrackConfiguration'])
    car=str(session.get('CarName',''))
    if (track,car)!=(TRACK_ATTENDU,VOITURE_ATTENDUE):
        raise RuntimeError('Session inattendue : '+track+'/'+car)
    if ACTIVER_SAUVEGARDE_CHECKPOINT:
        sauvegarder_checkpoint_si_absent(bridge,track,car)
    code,detail=bridge.request('STATUS')
    if code!='SAVED' or not ANCHOR_FILE.is_file():
        raise RuntimeError('Checkpoint absent. Immobiliser la voiture et activer '
                           'ACTIVER_SAUVEGARDE_CHECKPOINT dans REGLAGES puis Tout executer.')
    anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
    if (anchor.get('track'),anchor.get('car'))!=(track,car):
        raise RuntimeError('Reference de checkpoint incompatible avec la session.')
    if not ACTIVER_EVALUATION:
        print('Checkpoint pret. Evaluation desactivee, aucune conduite lancee.')
        return None
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    directory=PROJECT/'reports'/'evaluation_v3'/run_id
    directory.mkdir(parents=True,exist_ok=False)
    state={'session':run_id,'statut':'en_cours','debut_unix':time.time(),
           'deterministic':True,'apprentissage':False,'coupures_ppo':False,
           'episodes_demandes_par_modele':N_EPISODES,'track':track,'car':car,
           'curriculum_stage':ETAPE_APPRENTISSAGE,'episode_seconds':DUREE_EPISODE_S,
           'action_interval':INTERVALLE_ACTION_S,'offtrack_penalty':PENALITE_SORTIE_PISTE,
           'anchor':anchor,'modeles':[{'nom':d['label'],'path':str(d['path']),
               'sha256':d['sha256'],'meta_sha256':d['meta_sha256'],
               'compteur':d['meta']['total_timesteps']} for d in descriptors],
           'limites':['Meme depart fixe, repetitions deterministes : pas une mesure de generalisation.',
                      'Progression geometrique discretisee comme pendant entrainement.',
                      'Aucune collision detectee ne valide pas le detecteur.']}
    rows=[];core=None;active_row=None
    try:
        for model_index,(desc,agent) in enumerate(zip(descriptors,agents),1):
            print('Modele',model_index,'/',len(descriptors),':',desc['label'])
            core=LiveAssettoVisionEnv(track,car,episode_seconds=DUREE_EPISODE_S,
                 action_interval=INTERVALLE_ACTION_S,progress_sign=DIRECTION_PROGRESSION,
                 curriculum_stage=ETAPE_APPRENTISSAGE,offtrack_penalty=PENALITE_SORTIE_PISTE,
                 log_file=directory/('modele_%02d_steps.jsonl'%model_index))
            if agent.observation_space!=core.observation_space or agent.action_space!=core.action_space:
                raise RuntimeError('Espaces incompatibles : '+desc['label'])
            # No begin_rollout() : evaluation episodes have no PPO batch cuts.
            assert core.rollout_remaining is None
            for episode in range(1,N_EPISODES+1):
                active_row={'modele':desc['label'],'episode':episode,'valide':False,'succes':False,
                    'motif':'reset_en_cours','erreur':'','pas':0,'reward':0.,'progression_m':0.,
                    'duree_s':0.,'distance_parcourue_m':0.,'vitesse_max_kmh':0.,
                    'offtrack':False,'collision':False,'idle':False,'limite':False,
                    'volant_sature_pas':0,'variation_volant_totale':0.,'variations_comptees':0,
                    'ecart_lateral_max_normalise':0.,'camera_gap_max_s':0.}
                observation,reset_info=core.reset()
                if not reset_info.get('reset_verified'):
                    raise RuntimeError('Reset non verifie : evaluation refusee.')
                active_row.update(reset_distance_m=reset_info['reset_distance_m'],
                                  reset_yaw_error_rad=reset_info['reset_yaw_error_rad'])
                previous_xy=np.array(telemetry_xy(core.raw));previous_steer=None
                started=time.monotonic();done=False
                for _ in range(MAX_ACTIONS_PAR_EPISODE):
                    action,_=agent.predict(observation,deterministic=True)
                    observation,reward,terminated,truncated,info=core.step(action)
                    if info.get('sensor_error'):
                        raise RuntimeError('Capteurs indisponibles : '+str(info['sensor_error']))
                    if info.get('rollout_cut'):
                        raise RuntimeError('Coupure de lot inattendue pendant evaluation.')
                    active_row['pas']+=1;active_row['reward']+=float(reward)
                    active_row['progression_m']+=float(info.get('progress_m',0.))
                    active_row['vitesse_max_kmh']=max(active_row['vitesse_max_kmh'],float(info.get('speed_ms',0.))*3.6)
                    active_row['camera_gap_max_s']=max(active_row['camera_gap_max_s'],float(info.get('camera_gap_s',0.)))
                    active_row['ecart_lateral_max_normalise']=max(active_row['ecart_lateral_max_normalise'],abs(float(observation['state'][4])))
                    steer=float(info['steer_cmd'])
                    active_row['volant_sature_pas']+=int(abs(steer)>.95)
                    if previous_steer is not None:
                        active_row['variation_volant_totale']+=abs(steer-previous_steer)
                        active_row['variations_comptees']+=1
                    previous_steer=steer
                    xy=np.array(telemetry_xy(core.raw))
                    active_row['distance_parcourue_m']+=float(np.linalg.norm(xy-previous_xy));previous_xy=xy
                    for key in ('offtrack','collision','idle'):
                        active_row[key]=bool(active_row[key] or info.get(key,False))
                    active_row['succes']=bool(info.get('milestone_reached',False) or
                        (ETAPE_APPRENTISSAGE==4 and info.get('lap_finished',False)))
                    active_row['succes']=bool(active_row['succes'] and not active_row['offtrack'] and not active_row['collision'])
                    if terminated or truncated:
                        done=True
                        active_row['limite']=bool(truncated)
                        break
                core.safe_stop()
                active_row['duree_s']=time.monotonic()-started
                active_row['valide']=True
                if not done:active_row['limite']=True
                active_row['motif']=('sortie' if active_row['offtrack'] else
                    'collision_detectee' if active_row['collision'] else
                    'succes' if active_row['succes'] else 'immobilisation' if active_row['idle'] else 'limite')
                rows.append(active_row);active_row=None
                exporter_evaluation(directory,rows,state)
                print('Episode',episode,'/',N_EPISODES,':',rows[-1]['motif'],
                      '|',rows[-1]['pas'],'actions |',round(rows[-1]['duree_s'],2),'s')
            core.close();core=None
        state['statut']='terminee'
    except BaseException as exc:
        state['statut']='interrompue' if isinstance(exc,KeyboardInterrupt) else 'erreur_technique'
        state['erreur']=str(exc) or type(exc).__name__
        if active_row is not None:
            active_row['motif']='interruption' if isinstance(exc,KeyboardInterrupt) else 'erreur_technique'
            active_row['erreur']=state['erreur'];rows.append(active_row)
        (directory/'erreur.txt').write_text(traceback.format_exc(),encoding='utf8')
        raise
    finally:
        try:
            if core is not None:core.close()
        finally:
            state['fin_unix']=time.time()
            state['fichiers_modeles_inchanges']=all(file_sha256(d['path'])==d['sha256'] and
                file_sha256(d['meta_path'])==d['meta_sha256'] for d in descriptors)
            state['compteurs_inchanges']=all(int(a.num_timesteps)==int(d['meta']['total_timesteps'])
                                            for a,d in zip(agents,descriptors))
            summary=exporter_evaluation(directory,rows,state)
            print('Rapports conserves :',directory)
            display(summary)
    if not state['fichiers_modeles_inchanges'] or not state['compteurs_inchanges']:
        raise RuntimeError('Integrite des modeles modifiee pendant evaluation : voir etat_session.json.')
    return directory


## Executer la sequence

In [ ]:
RAPPORT_EVALUATION = executer_evaluation()
